# Phase 6c Wave 4: Hayden-Preskill QEC on MTC Horizon Substrate — Technical Notebook

Companion to `papers/paper35_qec_holography/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/QECHolographyBridge.lean` (10 substantive theorems / 0 sorry / 0 new axioms; verified `propext, Classical.choice, Quot.sound` only).

**Structure (mirrors paper §1–§6):**
1. Definitions: code distance $d_C := \log d_{\max}$ and scrambling-time bound $t_{\rm scr} := \log D^2$
2. Four MTC substrates: Fibonacci, Ising, an SU(3)$_{k=2}$ Fibonacci sub-sector witness, trivial abelian
3. The correctness-push biconditional $0 < d_C \Leftrightarrow 1 < d_{\max}$ and forward implication $0 < d_C \Rightarrow 0 < t_{\rm scr}$
4. Hayden-Preskill universal recovery at the scrambling-time bound
5. Concrete Fibonacci witness: in the universal-quantum-computation sense, Fibonacci is the smallest non-abelian MTC sufficient for fault-tolerant universal QC ($d_C = \log\varphi$, with $\log\varphi < \log 2$ as a quantitative reference). *Note*: Ising is admissible too and has *smaller* $d_C \approx \log\sqrt{2}$ — Fibonacci's "minimality" is a universal-QC literature claim (Nayak et al. 2008 §3), not a smallest-$d_C$ claim.
6. Trivial-abelian falsifier and W3 cross-bridge `horizon_BC_implies_HP_admissible`
7. Figure: code distance + scrambling-time bound across four substrates
8. Lean theorem inventory

All numerical content imports from `src.qec_holography` — no inline physics redefinition.

## 1. Definitions

Given a modular tensor category (MTC) of anyons with simple objects $\{a\}$ and per-anyon quantum dimensions $\{d_a\}$:

$$d_{\max} := \max_a d_a, \qquad D^2 := \sum_a d_a^2.$$

Two information-theoretic observables:

$$d_C := \log d_{\max} \quad (\text{code distance}), \qquad t_{\rm scr} := \log D^2 \quad (\text{Hayden-Preskill scrambling-time lower bound}).$$

$d_C$ measures topological-shielding scale; $t_{\rm scr}$ is the substrate-only contribution to $(β/2π) S_{BH}$ after the area law is subtracted. Mirrors of the Lean defs `HPCode.codeDistance` and `HPCode.scramblingTimeBound`.

In [ ]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.qec_holography import (
    MTCSpectrum,
    FIBONACCI_SPECTRUM,
    ISING_SPECTRUM,
    SU3K2_SPECTRUM,
    TRIVIAL_ABELIAN_SPECTRUM,
    code_distance,
    code_distance_admissible,
    global_dim_sq,
    scrambling_time_bound,
    recovery_threshold,
    recovery_possible_at_scrambling_bound,
)
from src.core.visualizations import COLORS, fig_code_distance_vs_fusion_spectrum

print('MTC observables across four substrates:')
print(f'  {"name":24s} | {"d_max":>7} | {"D²":>7} | {"d_C":>7} | {"t_scr":>7} | admissible')
print('  ' + '-'*72)
for name, sp in [
    ('Fibonacci',                 FIBONACCI_SPECTRUM),
    ('Ising',                     ISING_SPECTRUM),
    ('SU(3)_{k=2} Fib sub-sector', SU3K2_SPECTRUM),
    ('Trivial-Ab',                TRIVIAL_ABELIAN_SPECTRUM),
]:
    print(f'  {name:24s} | {sp.d_max:7.4f} | {global_dim_sq(sp):7.4f} | '
          f'{code_distance(sp):7.4f} | {scrambling_time_bound(sp):7.4f} | {code_distance_admissible(sp)}')

## 2. Four MTC substrates

**Fibonacci** $\{1, \tau\}$ with $d_\tau = \varphi = (1+\sqrt{5})/2 \approx 1.618$. $D^2 = 1 + \varphi^2 \approx 3.618$.

**Ising** $\{1, \sigma, \psi\}$ with $d_\sigma = \sqrt{2} \approx 1.414$. $D^2 = 1 + 2 + 1 = 4$.

**SU(3)$_{k=2}$ Fibonacci sub-sector**: a 2-element truncation $\{1, \text{adj}\}$ of the full SU(3)$_{k=2}$ MTC (which has 6 simple objects with $D^2 \approx 10.85$ — see `lean/SKEFTHawking/SU3kFusion.lean`). Within this *sub-sector* — which is not the full MTC — the spectrum is identical to Fibonacci ($d_{\max} = \varphi$), useful as an independent path to the same observables. The Python identifier `SU3K2_SPECTRUM` carries `name = "su3_k2_fibonacci_subsector"` for honesty.

**Trivial abelian** $\{1\}$. The vacuum-only MTC; $d_{\max} = 1$, $D^2 = 1$, $d_C = t_{\rm scr} = 0$. The canonical falsifier.

In [ ]:
for name, sp in [
    ('Fibonacci',                  FIBONACCI_SPECTRUM),
    ('Ising',                      ISING_SPECTRUM),
    ('SU(3)_{k=2} Fib sub-sector', SU3K2_SPECTRUM),
    ('Trivial-Ab',                 TRIVIAL_ABELIAN_SPECTRUM),
]:
    print(f'  {name:28s} spectrum (d_a) = {sp.quantum_dims}    name = {sp.name}')

## 3. Correctness-push biconditional

Lean theorem `code_distance_scaling_matches_anyonic_fusion_iff_fusion_in_admissible_class`:
$$(0 < d_C \;\Leftrightarrow\; 1 < d_{\max}) \;\wedge\; (0 < d_C \Rightarrow 0 < t_{\rm scr}).$$

The biconditional follows from `Real.log_pos_iff`. The forward implication follows from the sum bound $D^2 \geq d_{\max}^2$: if $d_{\max} > 1$ then $D^2 > 1$ so $t_{\rm scr} > 0$.

In [3]:
for name, sp in [
    ('Fibonacci',   FIBONACCI_SPECTRUM),
    ('Ising',       ISING_SPECTRUM),
    ('SU(3)_{k=2}', SU3K2_SPECTRUM),
    ('Trivial-Ab',  TRIVIAL_ABELIAN_SPECTRUM),
]:
    biconditional_lhs = code_distance(sp) > 0
    biconditional_rhs = sp.d_max > 1
    forward_implies = (code_distance(sp) <= 0) or (scrambling_time_bound(sp) > 0)
    print(f'  {name:12s}  0 < d_C: {biconditional_lhs}   1 < d_max: {biconditional_rhs}   '
          f'(0 < d_C → 0 < t_scr): {forward_implies}')

print()
print('  → all four substrates satisfy both halves of the correctness-push.')

  Fibonacci     0 < d_C: True   1 < d_max: True   (0 < d_C → 0 < t_scr): True
  Ising         0 < d_C: True   1 < d_max: True   (0 < d_C → 0 < t_scr): True
  SU(3)_{k=2}   0 < d_C: True   1 < d_max: True   (0 < d_C → 0 < t_scr): True
  Trivial-Ab    0 < d_C: False   1 < d_max: False   (0 < d_C → 0 < t_scr): True

  → all four substrates satisfy both halves of the correctness-push.


## 4. Hayden-Preskill universal recovery

Lean theorem `recovery_at_scrambling_bound`: for any encoding anyon $a$, recovery is possible at the scrambling-time bound, i.e., $\log d_a \leq \log D^2$. Proof case-splits on $d_a \geq 1$: if $d_a \geq 1$, $\log d_a \leq \log d_a^2 \leq \log D^2$; if $d_a < 1$, $\log d_a \leq 0 \leq \log D^2$.

In [4]:
import math
for name, sp in [
    ('Fibonacci',   FIBONACCI_SPECTRUM),
    ('Ising',       ISING_SPECTRUM),
    ('SU(3)_{k=2}', SU3K2_SPECTRUM),
    ('Trivial-Ab',  TRIVIAL_ABELIAN_SPECTRUM),
]:
    t_scr = scrambling_time_bound(sp)
    print(f'  {name:12s}  t_scr = {t_scr:.4f}')
    for idx, d_a in enumerate(sp.quantum_dims):
        thr = recovery_threshold(sp, idx)
        ok  = recovery_possible_at_scrambling_bound(sp, idx)
        print(f'      encoding idx={idx}  d_a={d_a:6.4f}  log d_a = {thr:7.4f}  recovery possible: {ok}')
    print()

  Fibonacci     t_scr = 1.2859
      encoding idx=0  d_a=1.0000  log d_a =  0.0000  recovery possible: True
      encoding idx=1  d_a=1.6180  log d_a =  0.4812  recovery possible: True

  Ising         t_scr = 1.3863
      encoding idx=0  d_a=1.0000  log d_a =  0.0000  recovery possible: True
      encoding idx=1  d_a=1.4142  log d_a =  0.3466  recovery possible: True
      encoding idx=2  d_a=1.0000  log d_a =  0.0000  recovery possible: True

  SU(3)_{k=2}   t_scr = 1.2859
      encoding idx=0  d_a=1.0000  log d_a =  0.0000  recovery possible: True
      encoding idx=1  d_a=1.6180  log d_a =  0.4812  recovery possible: True

  Trivial-Ab    t_scr = 0.0000
      encoding idx=0  d_a=1.0000  log d_a =  0.0000  recovery possible: True



## 5. Fibonacci as the smallest non-abelian admissible MTC for universal QC

Lean theorem `fibonacci_HPCode_codeDistance_lt_log_two`: $\log\varphi < \log 2$, i.e., the Fibonacci code distance sits *strictly below* the smallest power-of-2 quantum-dimension reference.

**Important caveat:** "Minimal" here is in the *universal-quantum-computation* sense (Nayak-Simon-Stern-Freedman-Das Sarma 2008 §3) — Fibonacci is the smallest non-abelian MTC whose braid group densely generates SU(2), making it the simplest topological substrate sufficient for fault-tolerant universal QC. **Ising has a smaller code distance** ($d_C = \log\sqrt{2} \approx 0.347 < \log\varphi \approx 0.481$) but Ising braiding is not universal — extra non-topological gates are needed. So "smallest non-abelian admissible MTC" depends on which sense of admissibility you intend.

In [ ]:
import math
phi = (1 + math.sqrt(5)) / 2
log_phi = math.log(phi)
log_two = math.log(2)
log_sqrt2 = math.log(math.sqrt(2))
print(f'  φ          = {phi:.6f}      (golden ratio)')
print(f'  log φ      = {log_phi:.6f}      (Fibonacci code distance d_C)')
print(f'  log √2     = {log_sqrt2:.6f}      (Ising code distance d_C — smaller than Fibonacci)')
print(f'  log 2      = {log_two:.6f}      (reference: smallest qubit code distance)')
print(f'  log φ < log 2:  {log_phi < log_two}')
print()
print('  Lean: fibonacci_HPCode_codeDistance_lt_log_two')
print('  → Fibonacci is the smallest non-abelian admissible MTC SUFFICIENT FOR UNIVERSAL QC.')
print('  → Ising has smaller code distance but its braiding is not universal alone.')

## 6. Trivial-abelian falsifier and W3 cross-bridge

**Falsifier**: Lean `trivialAbelian_violates_admissibility`. The trivial abelian MTC has $d_{\max} = 1$, so $d_C = \log 1 = 0$. The biconditional half $0 < d_C \Leftrightarrow 1 < d_{\max}$ then forces a contradiction $1 < 1$ — the substrate is structurally non-admissible.

**W3 cross-bridge**: Lean `horizon_BC_implies_HP_admissible`. If the W3 hypothesis bundle `H_HorizonBoundaryCondition` holds, then $1 < d_{\max}$ and admissibility follows. The proof body invokes `H_HorizonBoundaryCondition.areaLeading` and `Real.log_pos_iff`. **Honest disclosure**: the W3 structure has 5 fields (`areaLeading`, `positivity`, `secondLaw`, plus two placeholder `True` Props serving as documentation markers); only `areaLeading` is load-bearing in this proof. The "area-law / second-law / non-abelian envelope" prose paraphrase covers the substantive fields.

**Out of scope (deferred per Phase 6c roadmap §A):** AdS/CFT spectrum identification (specific holographic CFT for the horizon MTC), Yoshida-Kitaev decoder construction (we prove recovery is *possible* at the bound but don't construct the explicit decoding circuit), Page-curve quantitative reproduction.

In [6]:
sp = TRIVIAL_ABELIAN_SPECTRUM
print(f'  Trivial-abelian: d_max = {sp.d_max}   D² = {global_dim_sq(sp)}')
print(f'                   d_C   = {code_distance(sp)}    t_scr = {scrambling_time_bound(sp)}')
print(f'                   admissible = {code_distance_admissible(sp)}    (mirrors trivialAbelian_violates_admissibility)')

print()
print('  W3 cross-bridge (horizon_BC_implies_HP_admissible):')
print('  In Lean, the W3 H_HorizonBoundaryCondition.areaLeading field requires d_max > 1.')
print('  This forces admissibility for any horizon substrate satisfying the W3 hypothesis;')
print('  the Fibonacci instance witnesses non-vacuity (areaLawKappa > 0 in W3).')

  Trivial-abelian: d_max = 1.0   D² = 1.0
                   d_C   = 0.0    t_scr = 0.0
                   admissible = False    (mirrors trivialAbelian_violates_admissibility)

  W3 cross-bridge (horizon_BC_implies_HP_admissible):
  In Lean, the W3 H_HorizonBoundaryCondition.areaLeading field requires d_max > 1.
  This forces admissibility for any horizon substrate satisfying the W3 hypothesis;
  the Fibonacci instance witnesses non-vacuity (areaLawKappa > 0 in W3).


## 7. Figure — code distance and scrambling-time bound across substrates

**Left panel.** Code-distance bars across the four substrates with two reference lines: $d_C = 0$ (admissibility threshold) and $\log 2 \approx 0.693$ (Fibonacci upper bound). Fibonacci, Ising, and SU(3)$_{k=2}$ Fibonacci-subsector clear the threshold; trivial abelian sits at zero.

**Right panel.** Scrambling-time bound $t_{\rm scr}$ across the four substrates. The same admissibility pattern: three substrates above zero, trivial-abelian at zero. Demonstrates the correctness-push graphically.

In [7]:
# viz-ref: fig_code_distance_vs_fusion_spectrum
fig_code_distance_vs_fusion_spectrum().show()

## 8. Lean theorem inventory (10 substantive)

All theorems in `QECHolographyBridge.lean` are clean (zero `sorry`, zero new axioms; closure $\{\texttt{propext, Classical.choice, Quot.sound}\}$).

**Structures and definitions:**
- `HPCode` — pairs `HorizonMTCBC` with an encoding anyon index.
- `HPCode.codeDistance` — alias for `HorizonMTCBC.areaLawKappa`.
- `HPCode.scramblingTimeBound` := $\log D^2$.
- `HPCode.recoveryPossible B` := $\log d_a \leq B$.

**Substantive theorems:**
1. `one_le_globalDimSq` — $1 \leq D^2$ (unit object contribution).
2. `scramblingTimeBound_nonneg` — $0 \leq t_{\rm scr}$.
3. `codeDistance_nonneg` — $0 \leq d_C$ (inherited from W3).
4. `recovery_at_scrambling_bound` — universal Hayden-Preskill recovery.
5. `code_distance_scaling_matches_anyonic_fusion_iff_fusion_in_admissible_class` — correctness-push.
6. `nonabelian_anyon_implies_codeDistance_pos` — existence of $d_a > 1$ forces $d_C > 0$.
7. `fibonacci_HPCode_codeDistance_lt_log_two` — quantitative bound $\log \varphi < \log 2$.
8. `fibonacci_HPCode_scramblingTimeBound_pos` — Fibonacci scrambling-time positivity (calls W3 `fibonacci_horizon_areaLawKappa_pos`).
9. `trivialAbelian_violates_admissibility` — singleton-vacuum falsifier.
10. `horizon_BC_implies_HP_admissible` — substantive W3 cross-bridge.

Cross-wave strengthening (2026-04-28-0030) removed earlier API specializations `scramblingTimeBound_pos_iff_nontrivial` and `codeDistance_pos_iff_non_abelian` — content now lives inline in the correctness-push body.